In [21]:
# Qui importo le librerie utili
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import  LabelEncoder

In [22]:
napvscoffee = pd.read_csv("../power_nap_vs_coffee_effectiveness_dataset.csv")
#i ../ servono perché sono una cartella "indietro"
print(napvscoffee.head())

   participant_id  age            occupation  sleep_hours_previous_night  \
0               1   24  Working Professional                         5.4   
1               2   37               Student                         5.6   
2               3   32  Working Professional                         4.4   
3               4   28               Student                         6.9   
4               5   25  Working Professional                         4.7   

  intervention_type  intervention_duration_minutes  alertness_score_before  \
0         Power Nap                             15                      62   
1         Power Nap                             30                      67   
2            Coffee                             30                      44   
3            Coffee                             30                      59   
4         Power Nap                             30                      40   

   alertness_score_after  productivity_rating  mood_rating side_effects  


In [23]:
#controllo che non ci sono NaN, anche perchè con print(napvscoffee.head()) mi arrivano i NaN
napvscoffee.info()
napvscoffee.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   participant_id                 500 non-null    int64  
 1   age                            500 non-null    int64  
 2   occupation                     500 non-null    str    
 3   sleep_hours_previous_night     500 non-null    float64
 4   intervention_type              500 non-null    str    
 5   intervention_duration_minutes  500 non-null    int64  
 6   alertness_score_before         500 non-null    int64  
 7   alertness_score_after          500 non-null    int64  
 8   productivity_rating            500 non-null    int64  
 9   mood_rating                    500 non-null    int64  
 10  side_effects                   293 non-null    str    
dtypes: float64(1), int64(7), str(3)
memory usage: 43.1 KB


participant_id                     0
age                                0
occupation                         0
sleep_hours_previous_night         0
intervention_type                  0
intervention_duration_minutes      0
alertness_score_before             0
alertness_score_after              0
productivity_rating                0
mood_rating                        0
side_effects                     207
dtype: int64

Sorpresa! ci sono i NaN nei side effect >:( Dato che i NaN = no side effects io sarei per sostituirli con una bella stringa con scritto "No side effects" così che lo possiamo considerare nell'analisi.
In più dobbiamo considerare la questione se i side effect sono ordinali (ci sono side effects peggiori di altri?) o nominali (tutti i side effect sono più o meno uguali)

In [24]:
napvscoffee = napvscoffee.fillna("No side effects")
side_effects = napvscoffee["side_effects"]
print(set(side_effects))


{'Crash', 'Anxiety', 'No side effects', 'Grogginess'}


vabene, dobbiamo decidere: nel dubbio one-hot encodarli da meno bias, imo (anche perché come si decide qual è un side effect peggiore fra Grogginess, Anxiety e Crash?)

In [25]:
le = LabelEncoder()

napvscoffee_encoded = napvscoffee.copy()

napvscoffee_encoded["intervention_type"] = le.fit_transform(napvscoffee["intervention_type"])
napvscoffee_encoded = pd.get_dummies(napvscoffee_encoded, columns=["occupation"])

#se decidiamo di fare con i dummies
napvscoffee_encoded = pd.get_dummies(napvscoffee_encoded, columns=['side_effects'])
#napvscoffee_encoded= pd.Series(['No side effects', 'Grogginess', 'Anxiety', 'Crash'], name='side_effects')

print(napvscoffee_encoded.head())

   participant_id  age  sleep_hours_previous_night  intervention_type  \
0               1   24                         5.4                  1   
1               2   37                         5.6                  1   
2               3   32                         4.4                  0   
3               4   28                         6.9                  0   
4               5   25                         4.7                  1   

   intervention_duration_minutes  alertness_score_before  \
0                             15                      62   
1                             30                      67   
2                             30                      44   
3                             30                      59   
4                             30                      40   

   alertness_score_after  productivity_rating  mood_rating  \
0                     77                    5           10   
1                     83                    6            5   
2             

**To do**
- see the correlation
- scaling the data -> scale considerando lo split train/test
- drop the outliers
- why did you pick accurarcy/ recall -> consider the data

**Proposte**
- fare un decision three per vedere cosa predice meglio il caffè( il power nap (i sintomi? e.g.: crash e anxiety?)
- ma forse ha più senso predire i sintomi considerando le altre variabili, tipo quanto dormono a notte, l'età e il caffè vs power naps


In [2]:
napvscoffee_noID = napvscoffee_encoded.drop('participant_id', axis=1, inplace=False)
correlation = napvscoffee_noID.corr()

NameError: name 'napvscoffee_encoded' is not defined

In [38]:
threshold = 0.8

mask = np.abs(correlation) > threshold

mask

,age,sleep_hours_previous_night,intervention_type,intervention_duration_minutes,alertness_score_before,alertness_score_after,productivity_rating,mood_rating,occupation_Freelancer,occupation_Student,occupation_Working Professional,side_effects_Anxiety,side_effects_Crash,side_effects_Grogginess,side_effects_No side effects
age,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
sleep_hours_previous_night,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
intervention_type,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
intervention_duration_minutes,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
alertness_score_before,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False
alertness_score_after,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False
productivity_rating,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
mood_rating,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
occupation_Freelancer,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
occupation_Student,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False


from pandas.plotting import parallel_coordinates
parallel_coordinates(


In [36]:
#from pandas.plotting import parallel_coordinates
#noID_noocc = napvscoffee.drop(['participant_id', 'occupation'], axis=1)
#parallel_coordinates(noID, side_effects)
#plt.figure(figsize=(10,6))
#parallel_coordinates(encoded, 'side_effects')
#plt.show()

AttributeError: 'numpy.ndarray' object has no attribute 'columns'